In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/train'))
print(os.listdir('/content/drive/MyDrive/train'))

In [ ]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.init as init
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np
import math
from collections import Counter
import shutil
import os



torch.manual_seed(73)
np.random.seed(73)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------- model loader -------------------- #
# Run this section independently to load a saved model and classify new images.

def build_lora_model_for_loading(rank=4):
    m = models.resnet18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, 7)
    for param in m.parameters():
        param.requires_grad = False
    for name, module in m.layer4.named_modules():
        if isinstance(module, nn.Conv2d):
            parts  = name.split('.')
            parent = m.layer4
            for p in parts[:-1]:
                parent = getattr(parent, p)
            setattr(parent, parts[-1], LoRAConv2d(module, rank=rank))
    m.fc = LoRALinear(m.fc, rank=rank)
    return m

MODEL_PATH  = '/content/drive/MyDrive/main_model_best.pth' # set this to the model path
CLASS_NAMES = ['Glioblastoma', 'Metastatic', 'Schwannoma',
               'glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']

loaded_model = build_lora_model_for_loading(rank=4)
loaded_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
loaded_model = loaded_model.to(device)
loaded_model.eval()
print(f"Model loaded from {MODEL_PATH}")

loader_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def classify_image(image_path):
    from PIL import Image
    img    = Image.open(image_path).convert('RGB')
    tensor = loader_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(loaded_model(tensor), dim=1)[0]
        pred  = torch.argmax(probs).item()
    print(f"Predicted: {CLASS_NAMES[pred]}  (confidence: {probs[pred]:.2%})")
    for name, prob in zip(CLASS_NAMES, probs):
        print(f"  {name:<20} {prob:.2%}  {'#' * int(prob * 30)}")
    return CLASS_NAMES[pred]

print("Loader ready!")

# NOTE: You must set the model path in the code above

# To classify a new image call:
# classify_image('image_path_here.png')
# ex: classify_image('/content/drive/MyDrive/your_scan.jpg')

# NOTE: image must be 224x224 PNG file
